In [ ]:
import pandas as pd


In [ ]:
import pandas as pd


cols = ["seqname","source","feature","start","end","score","strand","frame","attribute"]

gtf = pd.read_csv(
    gtf_file,
    sep="\t",
    comment="#",      # skip header/comments that start with '#'
    header=None,      # GTFs typically have no header row
    names=cols,
    # nrows=100000,
    dtype=str,        # keep as strings; convert later if needed
    engine="python",
    na_filter=False
)

import re
# Parse attributes column into dicts


def parse_attr(attr_string):
    """
    Parse a GTF attribute string into a dict.
    - Most keys: single value (last occurrence wins, though usually unique)
    - 'tag': may appear multiple times -> list of all tags
    """
    parts = [p.strip() for p in attr_string.split(";") if p.strip()]
    d = {}
    tags = []

    for p in parts:
        # key "value"
        m = re.match(r'(\S+)\s+"([^"]+)"$', p)
        if m:
            key, value = m.groups()
            if key == "tag":
                tags.append(value)
            else:
                d[key] = value
            continue

        # key value  (unquoted, e.g. level 2)
        m2 = re.match(r'(\S+)\s+(\S+)$', p)
        if m2:
            key, value = m2.groups()
            if key == "tag":
                tags.append(value)
            else:
                d[key] = value

    if tags:
        d["tag"] = tags  # keep as list; you can stringify later if you want

    return d
    
    # Expand attribute dicts into dataframe columns
attr_df = gtf["attribute"].apply(parse_attr).apply(pd.Series)

# Merge back into df
gtf = pd.concat([gtf.drop(columns="attribute"), attr_df], axis=1)
gtf["is_MANE"] = gtf["tag"].apply(lambda x: "MANE_Select" in str(x))
gtf.to_csv("data/gencode.v49.annotation.gtf.csv", index=False)


In [ ]:
import os
import re
import uuid
import pandas as pd
from tqdm.auto import tqdm

def process_gtf_to_parquet(
    gtf_file: str,
    out_path: str,
    chunksize: int = 300_000,
    single_keys=None,
    keep_features=None,
    compression: str = "snappy",
):
    import pyarrow as pa
    import pyarrow.parquet as pq

    if single_keys is None:
        single_keys = [
            "gene_id", "gene_name", "gene_type",
            "transcript_id", "transcript_name", "transcript_type",
        ]

    cols = ["seqname","source","feature","start","end","score","strand","frame","attribute"]

    single_pat = {
        k: re.compile(rf'(?:^|;\s*){re.escape(k)}\s+"([^"]+)"')
        for k in single_keys
    }
    tag_pat = re.compile(r'(?:^|;\s*)tag\s+"([^"]+)"')

    def extract_single(attr_series: pd.Series, key: str) -> pd.Series:
        return attr_series.str.extract(single_pat[key], expand=False)

    def extract_tags(attr_series: pd.Series) -> pd.Series:
        return attr_series.apply(lambda s: tag_pat.findall(s))

    total_bytes = os.path.getsize(gtf_file)
    pbar = tqdm(total=total_bytes, unit="B", unit_scale=True, desc="Parsing GTF (approx)")

    # write to a unique temp file in same directory (so rename is atomic)
    out_dir = os.path.dirname(out_path) or "."
    os.makedirs(out_dir, exist_ok=True)
    tmp_path = os.path.join(out_dir, f".tmp_{uuid.uuid4().hex}_{os.path.basename(out_path)}")

    writer = None
    try:
        reader = pd.read_csv(
            gtf_file,
            sep="\t",
            comment="#",
            header=None,
            names=cols,
            dtype=str,
            engine="c",
            na_filter=False,
            chunksize=chunksize,
        )

        for chunk in reader:
            if keep_features is not None:
                chunk = chunk[chunk["feature"].isin(keep_features)]
                if chunk.empty:
                    continue

            attr = chunk["attribute"]

            for k in single_keys:
                chunk[k] = extract_single(attr, k)

            chunk["tag_list"] = extract_tags(attr)
            chunk["tag"] = chunk["tag_list"].apply(lambda x: "|".join(x) if x else "")
            chunk["is_MANE"] = chunk["tag_list"].apply(lambda x: "MANE_Select" in x)

            chunk = chunk.drop(columns=["attribute"])

            table = pa.Table.from_pandas(chunk, preserve_index=False)

            if writer is None:
                writer = pq.ParquetWriter(tmp_path, table.schema, compression=compression)

            writer.write_table(table)

            # progress: approximate by attribute length
            inc = int(attr.str.len().sum())
            pbar.update(min(inc, total_bytes - pbar.n))

    finally:
        if writer is not None:
            writer.close()
        pbar.close()

    # Now replace destination atomically
    # If out_path is locked, this will still fail, but now you won't lose tmp output.
    if os.path.exists(out_path):
        try:
            os.remove(out_path)
        except PermissionError:
            # fallback: write with a new name instead of failing
            alt = out_path.replace(".parquet", f".{uuid.uuid4().hex}.parquet")
            os.replace(tmp_path, alt)
            return alt

    os.replace(tmp_path, out_path)
    return out_path

process_gtf_to_parquet(
    gtf_file="annotations/gencode.v49.annotation.gtf.gz",
    out_path="data/gencode.v49.slim.parquet",
    chunksize=300_000)


Parsing GTF (approx):   0%|          | 0.00/93.4M [00:00<?, ?B/s]

'data/gencode.v49.slim.parquet'

In [6]:
gtf_pq = pd.read_parquet("data/gencode.v49.slim.parquet")
gtf_pq

,seqname,source,feature,start,end,score,strand,frame,gene_id,gene_name,gene_type,transcript_id,transcript_name,transcript_type,tag_list,tag,is_MANE
0,chr1,HAVANA,gene,11121,24894,.,+,.,ENSG00000290825.2,DDX11L16,lncRNA,None,None,None,[overlaps_pseudogene],overlaps_pseudogene,False
1,chr1,HAVANA,transcript,11121,14413,.,+,.,ENSG00000290825.2,DDX11L16,lncRNA,ENST00000832824.1,DDX11L16-260,lncRNA,[TAGENE],TAGENE,False
2,chr1,HAVANA,exon,11121,11211,.,+,.,ENSG00000290825.2,DDX11L16,lncRNA,ENST00000832824.1,DDX11L16-260,lncRNA,[TAGENE],TAGENE,False
3,chr1,HAVANA,exon,12010,12227,.,+,.,ENSG00000290825.2,DDX11L16,lncRNA,ENST00000832824.1,DDX11L16-260,lncRNA,[TAGENE],TAGENE,False
4,chr1,HAVANA,exon,12613,12721,.,+,.,ENSG00000290825.2,DDX11L16,lncRNA,ENST00000832824.1,DDX11L16-260,lncRNA,[TAGENE],TAGENE,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7750149,chrM,ENSEMBL,transcript,15888,15953,.,+,.,ENSG00000210195.2,MT-TT,Mt_tRNA,ENST00000387460.2,MT-TT-201,Mt_tRNA,"[basic, Ensembl_canonical, GENCODE_Primary]",basic|Ensembl_canonical|GENCODE_Primary,False
7750150,chrM,ENSEMBL,exon,15888,15953,.,+,.,ENSG00000210195.2,MT-TT,Mt_tRNA,ENST00000387460.2,MT-TT-201,Mt_tRNA,"[basic, Ensembl_canonical, GENCODE_Primary]",basic|Ensembl_canonical|GENCODE_Primary,False
7750151,chrM,ENSEMBL,gene,15956,16023,.,-,.,ENSG00000210196.2,MT-TP,Mt_tRNA,None,None,None,[],,False
7750152,chrM,ENSEMBL,transcript,15956,16023,.,-,.,ENSG00000210196.2,MT-TP,Mt_tRNA,ENST00000387461.2,MT-TP-201,Mt_tRNA,"[basic, Ensembl_canonical, GENCODE_Primary]",basic|Ensembl_canonical|GENCODE_Primary,False


In [ ]:
primary_genes = [
    "PSMB8", "PSMB9", "PSMB10",
    "PSME1", "PSME2", "PSME4",
    "TAP1", "TAP2", "TAPBP", "TAPBPL",
    "CALR", "PDIA3", "PDIA6",
    "B2M",
    "HLA-A", "HLA-B", "HLA-C", "HLA-E", "HLA-F", "HLA-G",
    "MICA", "MICB",
    "ULBP1", "ULBP2", "ULBP3", "RAET1E", "RAET1G", "RAET1L", # ULBP4, ULBP5, ULBP6
    "PVR", "NECTIN2",
    "SOCS1", "SOCS3",
    "NFKBIA", "TNFAIP3",
    "STAT1", "STAT3",
    "JAK1", "JAK2",
    "IRF1", "IRF2",
    "NLRC5",
    "CD274",          # PD-L1
    "PDCD1LG2",       # PD-L2
    "IFNG",
    "IFNGR1", "IFNGR2",
    "ADAM10", "ADAM17",
    "TGFB1",
    "IL6", "IL6R", "IL6ST",
    "PIAS1", "PIAS3",
    "PTPN2", "PTPN11",
    "EGFR",
    "SRC",
    "SMAD3",
    "EP300", "CREBBP"
]
genes = gtf[gtf["gene_name"].isin(primary_genes)]
genes.to_csv("data/primary_genes_all_features.csv", index=False)

In [ ]:
lncRNAs_genes = gtf[gtf["gene_type"] == "lncRNA"]
lncRNAs_genes = lncRNAs_genes[lncRNAs_genes["feature"] == "gene"]
lncRNAs_genes.to_csv("data/lncRNAs_all_features.csv", index=False)


In [10]:
genes = pd.read_csv("data/primary_genes_all_features.csv")


In [75]:
gtf[gtf["is_MANE"] == True]["tag"].iloc[0]

['RNA_Seq_supported_partial',
 'basic',
 'Ensembl_canonical',
 'GENCODE_Primary',
 'MANE_Select',
 'appris_principal_1',
 'CCDS']

In [35]:
gtf.loc[321131,"attribute"]

'gene_id "ENSG00000259030.8"; transcript_id "ENST00000533006.1"; gene_type "protein_coding"; gene_name "FPGT-TNNI3K"; transcript_type "protein_coding_CDS_not_defined"; transcript_name "FPGT-TNNI3K-204"; exon_number 5; exon_id "ENSE00002165375.1"; level 2; transcript_support_level "5"; hgnc_id "HGNC:42952"; tag "NMD_likely_if_extended"; tag "readthrough_transcript"; havana_gene "OTTHUMG00000166281.7"; havana_transcript "OTTHUMT00000388883.1";'

In [ ]:
gtf["is_MANE"]

tag
[basic, TAGENE]                                                                      189261
[basic, GENCODE_Primary, TAGENE]                                                      84980
[TAGENE]                                                                              65063
[basic, TAGENE, appris_principal_1, CCDS]                                             55248
[basic, TAGENE, appris_principal_3, CCDS]                                             36307
                                                                                      ...  
[low_sequence_quality, pseudo_consens, basic, Ensembl_canonical, GENCODE_Primary]         2
[polymorphic_pseudogene_no_stop]                                                          1
[readthrough_gene]                                                                        1
[fragmented_locus]                                                                        1
[ncRNA_host, overlaps_pseudogene, overlapping_locus]                        

In [56]:
vals = gtf[gtf["feature"] == "transcript"]["tag"].value_counts()
vals = vals.index
unique_vals = set()
for val in vals:
    for v in val:
        if v not in unique_vals:
            unique_vals.add(v)
unique_vals

{'3_nested_supported_extension',
 '3_standard_supported_extension',
 '454_RNA_Seq_supported',
 '5_nested_supported_extension',
 '5_standard_supported_extension',
 'CAGE_supported_TSS',
 'CCDS',
 'Ensembl_canonical',
 'GENCODE_Primary',
 'MANE_Plus_Clinical',
 'MANE_Select',
 'NAGNAG_splice_site',
 'NMD_exception',
 'NMD_likely_if_extended',
 'RNA_Seq_supported_only',
 'RNA_Seq_supported_partial',
 'RP_supported_TIS',
 'TAGENE',
 'alternative_3_UTR',
 'alternative_5_UTR',
 'appris_alternative_1',
 'appris_alternative_2',
 'appris_principal_1',
 'appris_principal_2',
 'appris_principal_3',
 'appris_principal_4',
 'appris_principal_5',
 'basic',
 'cds_end_NF',
 'cds_start_NF',
 'dotter_confirmed',
 'downstream_ATG',
 'exp_conf',
 'inferred_exon_combination',
 'inferred_transcript_model',
 'low_sequence_quality',
 'mRNA_end_NF',
 'mRNA_start_NF',
 'nested_454_RNA_Seq_supported',
 'non_ATG_start',
 'non_canonical_TEC',
 'non_canonical_U12',
 'non_canonical_conserved',
 'non_canonical_other'

In [62]:
gtf[gtf["feature"] == "UTR"]


,seqname,source,feature,start,end,score,strand,frame,gene_id,gene_type,...,transcript_name,exon_number,exon_id,transcript_support_level,havana_transcript,hgnc_id,havana_gene,ont,protein_id,ccdsid
2495,chr1,HAVANA,UTR,65419,65433,.,+,.,ENSG00000186092.7,protein_coding,...,OR4F5-201,1,ENSE00003812156.1,NaN,OTTHUMT00000003223.4,HGNC:14825,OTTHUMG00000001094.4,NaN,ENSP00000493376.2,CCDS30547.2
2496,chr1,HAVANA,UTR,65520,65564,.,+,.,ENSG00000186092.7,protein_coding,...,OR4F5-201,2,ENSE00003813641.1,NaN,OTTHUMT00000003223.4,HGNC:14825,OTTHUMG00000001094.4,NaN,ENSP00000493376.2,CCDS30547.2
2497,chr1,HAVANA,UTR,70006,71585,.,+,.,ENSG00000186092.7,protein_coding,...,OR4F5-201,3,ENSE00003813949.1,NaN,OTTHUMT00000003223.4,HGNC:14825,OTTHUMG00000001094.4,NaN,ENSP00000493376.2,CCDS30547.2
6563,chr1,HAVANA,UTR,450740,450742,.,-,.,ENSG00000284733.2,protein_coding,...,OR4F29-201,1,ENSE00003989331.1,NA,OTTHUMT00000007999.3,HGNC:31275,OTTHUMG00000002860.3,NaN,ENSP00000409316.1,CCDS72675.1
7178,chr1,HAVANA,UTR,685716,685718,.,-,.,ENSG00000284662.2,protein_coding,...,OR4F16-201,1,ENSE00004001351.2,NA,OTTHUMT00000007334.3,HGNC:15079,OTTHUMG00000002581.3,NaN,ENSP00000329982.2,CCDS41221.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
999899,chr2,HAVANA,UTR,99337378,99337554,.,+,.,ENSG00000158417.13,protein_coding,...,EIF5B-205,1,ENSE00004291612.1,NaN,NaN,HGNC:30793,OTTHUMG00000153242.3,NaN,ENSP00000526257.1,NaN
999900,chr2,HAVANA,UTR,99399412,99399776,.,+,.,ENSG00000158417.13,protein_coding,...,EIF5B-205,24,ENSE00004291610.1,NaN,NaN,HGNC:30793,OTTHUMG00000153242.3,NaN,ENSP00000526257.1,NaN
999946,chr2,HAVANA,UTR,99337388,99337554,.,+,.,ENSG00000158417.13,protein_coding,...,EIF5B-209,1,ENSE00004464500.1,NaN,NaN,HGNC:30793,OTTHUMG00000153242.3,NaN,ENSP00000638056.1,NaN
999947,chr2,HAVANA,UTR,99399412,99399780,.,+,.,ENSG00000158417.13,protein_coding,...,EIF5B-209,21,ENSE00004406566.1,NaN,NaN,HGNC:30793,OTTHUMG00000153242.3,NaN,ENSP00000638056.1,NaN


In [65]:
gtf[gtf["transcript_name"] == "OR4F5-201"]

,seqname,source,feature,start,end,score,strand,frame,gene_id,gene_type,...,transcript_name,exon_number,exon_id,transcript_support_level,havana_transcript,hgnc_id,havana_gene,ont,protein_id,ccdsid
2487,chr1,HAVANA,transcript,65419,71585,.,+,.,ENSG00000186092.7,protein_coding,...,OR4F5-201,NaN,NaN,NaN,OTTHUMT00000003223.4,HGNC:14825,OTTHUMG00000001094.4,NaN,ENSP00000493376.2,CCDS30547.2
2488,chr1,HAVANA,exon,65419,65433,.,+,.,ENSG00000186092.7,protein_coding,...,OR4F5-201,1,ENSE00003812156.1,NaN,OTTHUMT00000003223.4,HGNC:14825,OTTHUMG00000001094.4,NaN,ENSP00000493376.2,CCDS30547.2
2489,chr1,HAVANA,exon,65520,65573,.,+,.,ENSG00000186092.7,protein_coding,...,OR4F5-201,2,ENSE00003813641.1,NaN,OTTHUMT00000003223.4,HGNC:14825,OTTHUMG00000001094.4,NaN,ENSP00000493376.2,CCDS30547.2
2490,chr1,HAVANA,CDS,65565,65573,.,+,0,ENSG00000186092.7,protein_coding,...,OR4F5-201,2,ENSE00003813641.1,NaN,OTTHUMT00000003223.4,HGNC:14825,OTTHUMG00000001094.4,NaN,ENSP00000493376.2,CCDS30547.2
2491,chr1,HAVANA,start_codon,65565,65567,.,+,0,ENSG00000186092.7,protein_coding,...,OR4F5-201,2,ENSE00003813641.1,NaN,OTTHUMT00000003223.4,HGNC:14825,OTTHUMG00000001094.4,NaN,ENSP00000493376.2,CCDS30547.2
2492,chr1,HAVANA,exon,69037,71585,.,+,.,ENSG00000186092.7,protein_coding,...,OR4F5-201,3,ENSE00003813949.1,NaN,OTTHUMT00000003223.4,HGNC:14825,OTTHUMG00000001094.4,NaN,ENSP00000493376.2,CCDS30547.2
2493,chr1,HAVANA,CDS,69037,70005,.,+,0,ENSG00000186092.7,protein_coding,...,OR4F5-201,3,ENSE00003813949.1,NaN,OTTHUMT00000003223.4,HGNC:14825,OTTHUMG00000001094.4,NaN,ENSP00000493376.2,CCDS30547.2
2494,chr1,HAVANA,stop_codon,70006,70008,.,+,0,ENSG00000186092.7,protein_coding,...,OR4F5-201,3,ENSE00003813949.1,NaN,OTTHUMT00000003223.4,HGNC:14825,OTTHUMG00000001094.4,NaN,ENSP00000493376.2,CCDS30547.2
2495,chr1,HAVANA,UTR,65419,65433,.,+,.,ENSG00000186092.7,protein_coding,...,OR4F5-201,1,ENSE00003812156.1,NaN,OTTHUMT00000003223.4,HGNC:14825,OTTHUMG00000001094.4,NaN,ENSP00000493376.2,CCDS30547.2
2496,chr1,HAVANA,UTR,65520,65564,.,+,.,ENSG00000186092.7,protein_coding,...,OR4F5-201,2,ENSE00003813641.1,NaN,OTTHUMT00000003223.4,HGNC:14825,OTTHUMG00000001094.4,NaN,ENSP00000493376.2,CCDS30547.2


In [ ]:
PAM50.lo`c[PAM50["sample_id"].str.split("-").str[-1] == "11"]

,sample_short,sample_id,PAM50_recomputed,sampleID,PAM50Call_RNAseq,PAM50_final,thornsson_subtype
126,TCGA-A7-A0CE-11,TCGA-A7-A0CE-11,Normal,TCGA-A7-A0CE-11,NaN,Normal,NaN
129,TCGA-A7-A0CH-11,TCGA-A7-A0CH-11,LumA,TCGA-A7-A0CH-11,NaN,Normal,NaN
132,TCGA-A7-A0D9-11,TCGA-A7-A0D9-11,Normal,TCGA-A7-A0D9-11,Normal,Normal,NaN
135,TCGA-A7-A0DB-11,TCGA-A7-A0DB-11,Basal,TCGA-A7-A0DB-11,Normal,Normal,NaN
137,TCGA-A7-A0DC-11,TCGA-A7-A0DC-11,Normal,TCGA-A7-A0DC-11,Normal,Normal,NaN
...,...,...,...,...,...,...,...
1032,TCGA-E9-A1RF-11,TCGA-E9-A1RF-11,LumA,TCGA-E9-A1RF-11,LumA,Normal,NaN
1035,TCGA-E9-A1RH-11,TCGA-E9-A1RH-11,Normal,TCGA-E9-A1RH-11,Normal,Normal,NaN
1037,TCGA-E9-A1RI-11,TCGA-E9-A1RI-11,LumA,TCGA-E9-A1RI-11,LumA,Normal,NaN
1113,TCGA-GI-A2C8-11,TCGA-GI-A2C8-11,LumA,TCGA-GI-A2C8-11,LumA,Normal,NaN
